In [1]:
# Code adapted from Machine Learning Engineering (Cornell Tech 2025)
import torch
import numpy as np
import random

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [2]:
from google.colab import drive
import sys
import os

# --- 1. Mount Drive ---
drive.mount('/content/drive')

# --- 2. Define Paths ---
# Path to the source code (loaders.py) - REMAINS ON DRIVE
DRIVE_CODE_PATH = '/content/drive/MyDrive/GoogleColab/dataforsptransformer/'

# Path to the zipped data file on Drive
ZIP_SOURCE_PATH = os.path.join(DRIVE_CODE_PATH, 'crc_markers.zip')

# Local disk folder where the FAST images will be unzipped
FAST_DATA_PATH = '/content/fast_data/'

# --- 3. Unzip Data (Performance Fix) ---
if not os.path.exists(FAST_DATA_PATH):
    print(f"🚀 Unzipping data from Drive to fast local disk: {FAST_DATA_PATH}")
    !mkdir -p "$FAST_DATA_PATH"
    # The -q flag silences the output. -d sets the destination directory.
    !unzip -q "$ZIP_SOURCE_PATH" -d "$FAST_DATA_PATH"

    print("✅ Data transfer complete. Starting new batch load test.")
else:
    print("Fast data directory already exists.")


# --- 4. Set Final Variables ---
# PROJECT_DIR for the rest of your notebook now points to the FAST images
PROJECT_DIR = FAST_DATA_PATH

# Add the Drive path for Python to find 'loaders.py' and other modules
if DRIVE_CODE_PATH not in sys.path:
    sys.path.append(DRIVE_CODE_PATH)
    print(f"✅ Added {DRIVE_CODE_PATH} to Python system path.")

In [5]:
from google.colab import files

# This will open a 'Choose Files' button in your output cell
# Choose train_helpers.py
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [6]:
!python3 data_cleaning.py

In [7]:
import pandas as pd

df = pd.read_csv("/content/fast_data/CRC_clusters_neighborhoods_markers_cleaned.csv")
df.head()

In [8]:
!pip install torch-geometric

In [9]:
df.columns

## Setup Training Environment

In [ ]:
# Add current directory to path so we can import training.py
import sys
import os
sys.path.append(os.getcwd())

from training import train_graphformer, simple_neighbor_average_baseline

# Start and end protein columns as identified in the notebook
START_PROTEIN = 'CD44 - stroma:Cyc_2_ch_2'
END_PROTEIN = 'CD138 - plasma cells:Cyc_21_ch_3'
protein_cols = df.columns[df.columns.get_loc(START_PROTEIN) : df.columns.get_loc(END_PROTEIN)+1].tolist()

## Level 1 Hyperparameter Optimization

We optimize the core capacity and stability parameters first:
1. **Learning Rate**: Determines convergence speed and stability.
2. **Max Neighbors (k)**: Determines the biological spatial context size.
3. **Number of Layers**: Determines the depth of the graph receptive field.

In [ ]:
import itertools

# Level 1 Hyperparameter Grid
hparams = {
    'lr': [1e-3, 1e-4, 5e-5],
    'max_neighbors': [5, 10, 20],
    'num_layers': [2, 3, 4]
}

results = []

keys, values = zip(*hparams.items())
for v in itertools.product(*values):
    params = dict(zip(keys, v))
    
    print(f"\nTesting Config: {params}")
    
    # Run a short training burst (e.g., 5-10 epochs) for faster sweeping
    model, train_ds, test_ds = train_graphformer(
        df,
        protein_start_col=START_PROTEIN,
        protein_end_col=END_PROTEIN,
        normalize=True,
        hidden_dim=128,  # Fixed for Level 1
        num_heads=4,     # Fixed for Level 1
        num_epochs=10,   # Shorter training for optimization
        **params
    )
    
    # Evaluate performance (using final R2 from training function logs or manual retrieval)
    # Note: train_graphformer returns the best model based on R2 internally
    import torch
    checkpoint = torch.load('best_graphformer_model.pt', weights_only=False)
    best_r2_val = checkpoint['r2']
    
    results.append({**params, 'best_r2': best_r2_val})
    print(f"Config {params} achieved best R2: {best_r2_val:.4f}")

# Display and sort results
results_df = pd.DataFrame(results).sort_values(by='best_r2', ascending=False)
print("\nHyperparameter Optimization Results (Sorted by R2):")
print(results_df)